# Bechmarks of timestep used

In [ ]:
# import needed packages

#update reading in packages when rerunning this cell
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/nethome/4291387/Maxey_Riley_advection/Maxey_Riley_advection/src")
import numpy as np
import xarray as xr 
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from mpl_toolkits.axes_grid1 import Divider, Size
import matplotlib.cm as cm
from matplotlib import colormaps
import cartopy.crs as ccrs #for plotting on map
import cartopy as cart
from datetime import datetime, timedelta
from importlib import reload

from analysis_functions_xr import trajectory_length, calc_tidal_av, skill_score
from analysis_functions import make_PDF, Haversine, make_lognormal_PDF
from particle_characteristics_functions import factor_drag_white1991, dynamic_viscosity_Sharqawy, stokes_relaxation_time, buoyancy_drifter, Re_particle

plt.style.use('../python_style_Meike.mplstyle')

Viridis = colormaps['viridis']
Viridislist = Viridis(np.linspace(0, 0.95, 5))#Magma(np.linspace(0.2, 0.9, 5))


In [ ]:
# set needed constants
Rearth = 6371 * 10**3 # in m,
deg2rad = np.pi / 180.
sec_in_hours= 3600
diameter = 0.25 #m
diameter_in =0.2
heigth_drifter = 0.041  # m (heigth drifter)
m_drifter = 0.905 # kg (mass drifter) 
rho_water = 1027 # kg/m3 
av_temp_NWES = 11.983276 # [degrees C]
av_salinity_NWES = 34.70449/1000 # [kg/kg]
dynamic_viscosity_water = dynamic_viscosity_Sharqawy(T = av_temp_NWES, S = av_salinity_NWES) # kg/(ms) https://www.engineeringtoolbox.com/sea-water-properties-d_840.html (at 10 deg)
kinematic_viscosity_water = dynamic_viscosity_water / rho_water
B = buoyancy_drifter(diameter  = diameter_in, heigth = heigth_drifter, mass = m_drifter, density_fluid = rho_water)
tau = stokes_relaxation_time(diameter = diameter, kinematic_viscosity = kinematic_viscosity_water, buoyancy = B)


In [ ]:
Replist = np.array([100,450,1000]) # list of fixed input particle reynolds numbers used
coriolis = True 
gradient = True
# round numbers
B = round(B,2)
tau = int(tau)
runtime =  timedelta(days=30)# timedelta(days=10)
land_handling = 'delete_beaching'
nparticles = 52511 # number of particles
chunck_time = 100 # chuncking used in time
loc = 'NWES' # release location
Tmax=720 # number of timesteps
freqmin = 60
dtsteps = [0,1, 5, 15 , 30]
dtstepsnames = ['30 s', '1 min','5 min','15 min','30 min']

starttime = datetime(2023, 9, 1, 0, 0, 0, 0)

base_directory = '/storage/shared/oceanparcels/output_data/data_Meike/MR_advection/NWES/'

basefile_Rep_constant = (base_directory + '{particle_type}/{loc}_'
                 'start{y_s:04d}_{m_s:02d}_{d_s:02d}_'
                 'end{y_e:04d}_{m_e:02d}_{d_e:02d}_RK4_'
                 '_Rep_{Rep:04d}_B{B:04d}_tau{tau:04d}_{land_handling}_cor_{coriolis}_gradient_{gradient}_writefreq{freqmin:02d}min.zarr') 

basefile_MR_Rep_drag = (base_directory + '{particle_type}/{loc}_start{y_s:04d}_{m_s:02d}_{d_s:02d}'
                 '_end{y_e:04d}_{m_e:02d}_{d_e:02d}_RK4_B{B:04d}_tau{tau:04d}_{land_handling}_cor_{coriolis}_gradient_{gradient}_writefreq{freqmin:02d}min.zarr') 



particle_types = ['inertial_SM_drag_Rep','inertial_SM_Rep_constant']


basefiles = {'inertial_SM_Rep_constant':basefile_Rep_constant,
             'inertial_SM_drag_Rep':basefile_MR_Rep_drag
             }

Repvalues = {'inertial_SM_drag_Rep':[None],
             'inertial_SM_Rep_constant':[0]}

In [ ]:
# read in data
endtime = starttime + runtime 
date = f'{starttime.year:04d}/{starttime.month:02d}'

data ={}
for pt in particle_types:
    data[pt]={}
    for dtstep in dtsteps:
        data[pt][dtstep]={}

        for Rep in Repvalues[pt]:
            data[pt][dtstep][Rep]={}

for pt in particle_types:
     for dtstep in dtsteps:
   
            
            
            for Rep in Repvalues[pt]:
                file = basefiles[pt].format(loc=loc,
                                            y_s=starttime.year,
                                            m_s=starttime.month,
                                            d_s=starttime.day,
                                            y_e=endtime.year,
                                            m_e=endtime.month,
                                            d_e=endtime.day,
                                            B = int(B * 1000), 
                                            tau = int(tau ),
                                            land_handling = land_handling, 
                                            coriolis = coriolis,
                                            gradient = gradient,
                                            particle_type = pt,
                                            Rep = Rep, 
                                            freqmin = dtstep)
                ds = xr.open_dataset(file,
                                    engine='zarr',
                                    chunks={'trajectory':nparticles, 'obs':chunck_time},
                                    drop_variables=['B','tau','z'],
                                    decode_times=False) 
                # if(pt == 'inertial_SM_drag_Rep'):
                Uslip = np.sqrt(ds.uslip**2 + ds.vslip**2) 
                ds = ds.assign(Uslip = Uslip)
                Rep_measured = Re_particle(Uslip,diameter, kinematic_viscosity_water)
                ds = ds.assign(Rep_measured = Rep_measured)
   
                data[pt][dtstep][Rep][date]= ds 


In [ ]:
#fig 4(a)
# plot selection of trajectories
id =12

pt = 'inertial_SM_drag_Rep'#'inertial_SM_Rep_constant'
Rep = None

fig = plt.figure()
h = [Size.Fixed(1.0), Size.Fixed(8)]
v = [Size.Fixed(0.7), Size.Fixed(6)]
divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)
ax = fig.add_axes(divider.get_position(),
                  axes_locator=divider.new_locator(nx=1, ny=1),projection=ccrs.PlateCarree())
colorlist=['k','cornflowerblue','orange','purple']
markers = ['-','--','-.',':']
year=2023
month=9
date = f'{year:04d}/{month:02d}'
legend= []

ax.coastlines()
ax.add_feature(cart.feature.LAND, facecolor='lightgrey')
for dtstep, color in zip(dtsteps,Viridislist):
    ax.plot(data[pt][dtstep][Rep][date].lon[id,0::1].T,
    data[pt][dtstep][Rep][date].lat[id,0::1].T,
    '-',
    color=color,zorder=20);



ax.legend(dtstepsnames)
# add 5 day markers
# for dtstep, color, marker in zip(particle_types,colorlist,markers):
#     ax.plot(ddata[pt][dtstep][Rep][date].lon[id,5*24::5*24].T,
#                                 data[pt][dtstep][Rep][date].lat[id,5*24::5*24].T,
#                                 'o',
#                                 color=color);

gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
        linewidth=0, color='gray', alpha=0.5, linestyle='-')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 18}
gl.ylabel_style =  {'size': 18}

# ax.set_xlim(-14.6,-11.1)
# ax.set_ylim(54.4,56)

ax.plot(data[pt][dtstep][Rep][date].lon[id,0].T,
data[pt][dtstep][Rep][date].lat[id,0].T,
'>',
color='darkgreen',zorder=20,markersize=15);
ax.set_title('timesteps',fontsize=18)




In [ ]:
# along track distance
for pt in particle_types: # loop over all particle types
    for dtstep in dtsteps:
        for Rep in Repvalues[pt]:
            ds =data[pt][dtstep][Rep][date]
            traj_length = trajectory_length(ds.lon, ds.lat)
            data[pt][dtstep][Rep][date] = ds.assign(traj_length = traj_length)

In [ ]:
#  plot PDF of total trajectory length
fig,ax=plt.subplots()

Tmax = 400

legend=[]
pt = 'inertial_SM_Rep_constant'
Rep = 0
for dtstep,color in zip([1,5,15,30],Viridislist[1:]):
    array = (data[pt][dtstep][Rep][date].traj_length[:,Tmax-1]).values.flatten()
    array = array[~np.isnan(array)]
    bins, pdf = make_PDF( array,501, norm=True,vmin=0,vmax=3000)#,vmin = -5,vmax = 5)
    ax.plot(bins,pdf,'-',color=color)#,alpha=0.2)
    
ax.legend(dtstepsnames[1:])
ax.set_ylabel("PDF")
ax.set_xlabel("trajectory length (km)")

In [ ]:
#  plot PDF of total trajectory length
fig,ax=plt.subplots()

Tmax = 400

legend=[]
pt = 'inertial_SM_drag_Rep'
Rep = None

#ref
array = (data[pt][0][Rep][date].traj_length[:,Tmax-1]).values.flatten()
array = array[~np.isnan(array)]
bins1, pdf1 = make_PDF( array,51, norm=True,vmin=0,vmax=3000)#,vmin = -5,vmax = 5)

for dtstep, color in zip([1,5,15,30],Viridislist[1:]):
    array = (data[pt][dtstep][Rep][date].traj_length[:,Tmax-1]).values.flatten()
    array = array[~np.isnan(array)]
    bins, pdf = make_PDF( array,51, norm=True,vmin=0,vmax=3000)#,vmin = -5,vmax = 5)
    ax.plot(bins,pdf-pdf1,'-',color=color)
    
ax.legend(dtstepsnames[1:])
ax.set_ylabel("PDF")
ax.set_xlabel("trajectory length (km)")

#  plot PDF of total trajectory length
fig,ax=plt.subplots()

Tmax = 400

legend=[]
pt = 'inertial_SM_Rep_constant'
Rep = 0

#ref
array = (data[pt][0][Rep][date].traj_length[:,Tmax-1]).values.flatten()
array = array[~np.isnan(array)]
bins1, pdf1 = make_PDF( array,51, norm=True,vmin=0,vmax=3000)#,vmin = -5,vmax = 5)

for dtstep, color in zip([1,5,15,30],Viridislist[1:]):
    array = (data[pt][dtstep][Rep][date].traj_length[:,Tmax-1]).values.flatten()
    array = array[~np.isnan(array)]
    bins, pdf = make_PDF( array,51, norm=True,vmin=0,vmax=3000)#,vmin = -5,vmax = 5)
    ax.plot(bins,pdf-pdf1,'-',color=color)
    
ax.legend(dtstepsnames[1:])
ax.set_ylabel("PDF")
ax.set_xlabel("trajectory length (km)")

In [ ]:
#  plot PDF of total trajectory length
fig,ax=plt.subplots()

Tmax = 400

legend=[]
pt = 'inertial_SM_Rep_constant'
Rep = 0
for dtstep, color in zip([1,5,15,30],Viridislist[1:]):
    array = (data[pt][dtstep][Rep][date].traj_length[:,Tmax-1]-data[pt][0][Rep][date].traj_length[:,Tmax-1]).values.flatten()
    array = array[~np.isnan(array)]
    bins, pdf = make_PDF( array,201, norm=True,vmin = -2,vmax = 2)
    ax.plot(bins,pdf,'-',color=color)
    
ax.legend(dtstepsnames[1:])
ax.set_ylabel("PDF")
ax.set_xlabel("trajectory length (km)")

In [ ]:
for pt in particle_types:
    for Rep in Repvalues[pt]:
        ds_30s = data[pt][0][Rep][date]
        for dtstep in [1,5,15,30]:
                ds_dtstep = data[pt][dtstep][Rep][date]
                dist= Haversine(ds_dtstep['lon'],ds_dtstep['lat'],
                                    ds_30s['lon'],ds_30s['lat'])
                data[pt][dtstep][Rep][date]=data[pt][dtstep][Rep][date].assign(dist = dist)
                data[pt][dtstep][Rep][date]=data[pt][dtstep][Rep][date].rename({'dist':'dist_to_30s'})






In [ ]:
pt = 'inertial_SM_drag_Rep'
Rep = None

Tmax=400
tlist=np.arange(0,Tmax,1)/24



fig, ax = plt.subplots()
for dtstep, color in zip([1,5,15,30],Viridislist[1:]):
    ax.plot(tlist,data[pt][dtstep][None][date]['dist_to_30s'].mean(dim='trajectory',skipna=True)[0:Tmax],'-',color=color)
    ax.plot(tlist,data[pt][dtstep][None][date]['dist_to_30s'].median(dim='trajectory',skipna=True)[0:Tmax],'--',color=color,label='_nolegend_')

ax.legend(dtstepsnames[1:])

In [ ]:
pt = 'inertial_SM_Rep_constant'
Rep = 0

Tmax=700
tlist=np.arange(0,Tmax,1)/24



fig, ax = plt.subplots()
# ax2 = ax.twinx()
for dtstep, color in zip([1,5,15,30],Viridislist[1:]):
    ax.plot(tlist,data[pt][dtstep][0][date]['dist_to_30s'].mean(dim='trajectory',skipna=True)[0:Tmax],'-',color=color)
    # ax2.plot(tlist,data[pt][dtstep][0][date]['dist_to_30s'].median(dim='trajectory',skipna=True)[0:Tmax],'--',color=color,label='_nolegend_')
ax.legend(dtstepsnames[1:])

ax.set_ylabel('Relative distance [km] to dt = 30 s')
ax.set_xlabel('time [days]')
ax.set_title('SM-MRG, Re$_p$ = 0',fontsize=18)


In [ ]:
#  plot PDF of total trajectory length
fig,ax=plt.subplots()

Tmax = 400

legend=[]
pt = 'inertial_SM_Rep_constant'
Rep = 0
for dtstep, color in zip([1,5,15,30],Viridislist[1:]):
    array = (data[pt][dtstep][Rep][date].dist_to_30s[:,Tmax-1]).values.flatten()
    array = array[~np.isnan(array)]
    bins, pdf = make_PDF( array,250, norm=False,vmin = 0,vmax=250)#,vmax = 5)
    ax.plot(bins,pdf/nparticles,'-',color=color)
    
ax.legend(dtstepsnames[1:])
ax.set_ylabel("fraction of particles")
ax.set_xlabel("final separation distance(km)")
ax.set_xscale('log')
ax.set_yscale('log')

In [ ]:
pt = 'inertial_SM_drag_Rep'#'inertial_SM_Rep_constant'
Rep = None
nt=400
for dtstep in dtsteps[1:]:
    fig = plt.figure()
    h = [Size.Fixed(1.0), Size.Fixed(8)]
    v = [Size.Fixed(0.7), Size.Fixed(6)]
    divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)
    ax = fig.add_axes(divider.get_position(),
                    axes_locator=divider.new_locator(nx=1, ny=1),projection=ccrs.PlateCarree())
    colorlist=['k','cornflowerblue','orange','purple']
    markers = ['-','--','-.',':']
    year=2023
    month=9
    date = f'{year:04d}/{month:02d}'
    legend= []

    ax.coastlines()
    ax.add_feature(cart.feature.LAND, facecolor='lightgrey')

    sc = ax.scatter(data[pt][dtstep][Rep][date].lon[:,0],
    data[pt][dtstep][Rep][date].lat[:,0],c =data[pt][dtstep][Rep][date].dist_to_30s[:,nt],cmap='viridis',s=0.6,vmin=0,vmax=10);
    fig.colorbar(sc,fraction=0.03,extend='max')


In [ ]:
pt = 'inertial_SM_Rep_constant'#'inertial_SM_Rep_constant'
Rep = 0
nt=400
for dtstep in dtsteps[1:]:
    fig = plt.figure()
    h = [Size.Fixed(1.0), Size.Fixed(8)]
    v = [Size.Fixed(0.7), Size.Fixed(6)]
    divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)
    ax = fig.add_axes(divider.get_position(),
                    axes_locator=divider.new_locator(nx=1, ny=1),projection=ccrs.PlateCarree())
    colorlist=['k','cornflowerblue','orange','purple']
    markers = ['-','--','-.',':']
    year=2023
    month=9
    date = f'{year:04d}/{month:02d}'
    legend= []

    ax.coastlines()
    ax.add_feature(cart.feature.LAND, facecolor='lightgrey')

    sc = ax.scatter(data[pt][dtstep][Rep][date].lon[:,0],
    data[pt][dtstep][Rep][date].lat[:,0],c =data[pt][dtstep][Rep][date].dist_to_30s[:,nt],cmap='viridis',s=0.6,vmin=0,vmax=50);
    fig.colorbar(sc,fraction=0.03,extend='max')
